# 04 — Hazard Mechanism and State Dynamics (Figure 4)

**Question**: How does collective order modulate the instantaneous probability that a
centroid run ends?  Can a Markov order-state model with age-dependent killing
reproduce the empirical run-duration distribution?

## Panels
| Panel | Content |
|-------|---------|
| **A** | Empirical hazard $h(a)$ vs run age — inverse-age fit |
| **B** | Hazard split by order state (low/mid/high) |
| **C** | Markov state transition matrix $P$ |
| **D** | Empirical vs. simulated run-duration CCDF |
| **E** | Hazard ratios from Cox model (order + speed) |

**Key result**: Order suppresses hazard; age-dependent killing + state dynamics
explains heavy-tailed run durations.

**Outstanding TODO**: Bootstrap confidence intervals on hazard ratios.
AIC/BIC comparison (baseline vs. order-dependent model).

**Prerequisite**: Run `01_data_loading.ipynb` first.

In [ ]:
import sys, os
from pathlib import Path

_HERE = Path(os.getcwd())
_REPO = _HERE.parents[2]
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import chi2

from analysis.levy_paper.util.paper_utils import (
    configure_paper_plotting,
    load_cache, save_figure,
    plot_ccdf, plot_hazard,
    plot_state_transition_matrix,
    STATE_COLORS, STATE_LABELS, A0, N_SIM,
)
configure_paper_plotting()

# ── Load caches ────────────────────────────────────────────────────────────
hazard_intervals = load_cache("hazard_intervals")
centroid_runs    = load_cache("centroid_order_runs")

print(f"Hazard intervals: {len(hazard_intervals):,}")
print(hazard_intervals.dtypes)

## Panel A — Empirical Hazard vs Run Age

Estimate $h(a)$ as the fraction of at-risk runs that terminate within each
age-bin interval, with Poisson confidence intervals.

In [ ]:
# Compute empirical hazard by age bin
haz_agg = (
    hazard_intervals
    .groupby("age_bin")["event"]
    .agg(events="sum", at_risk="count")
    .assign(
        h=lambda d: d["events"] / d["at_risk"],
        # Poisson 95% CI: ±1.96 * sqrt(n_events) / n_at_risk
        h_lo=lambda d: np.maximum(0, (d["events"] - 1.96 * np.sqrt(d["events"])) / d["at_risk"]),
        h_hi=lambda d: (d["events"] + 1.96 * np.sqrt(d["events"])) / d["at_risk"],
    )
    .reset_index()
)

age_centres = haz_agg["age_bin"].values.astype(float)

fig_A, ax_A = plt.subplots(figsize=(4.5, 3.5))
params = plot_hazard(
    ax_A,
    age_centres, haz_agg["h"].values,
    ci_lo=haz_agg["h_lo"].values,
    ci_hi=haz_agg["h_hi"].values,
    label="Empirical",
    color="black",
    fit_inverse_age=True,
    a0=A0,
)
ax_A.set_title("A — Empirical hazard vs age")
ax_A.legend(fontsize=8)
print("Baseline fit:", params)
plt.tight_layout()
plt.show()

## Panel B — Hazard Split by Collective Order State

Does high-order suppress termination risk at each age?

In [ ]:
fig_B, ax_B = plt.subplots(figsize=(4.5, 3.5))

state_params = {}
for state in STATE_LABELS:
    sub = hazard_intervals.loc[hazard_intervals["p_state"] == state]
    haz_s = (
        sub.groupby("age_bin")["event"]
        .agg(events="sum", at_risk="count")
        .assign(h=lambda d: d["events"] / d["at_risk"])
        .reset_index()
    )
    p = plot_hazard(
        ax_B,
        haz_s["age_bin"].values.astype(float),
        haz_s["h"].values,
        label=state.capitalize(),
        color=STATE_COLORS[state],
        fit_inverse_age=True,
        a0=A0,
    )
    state_params[state] = p

ax_B.set_title("B — Hazard by order state")
ax_B.legend(fontsize=8)
print("State-specific fits:", state_params)
plt.tight_layout()
plt.show()

## Panel C — State Transition Matrix

Treat the order state sequence as a discrete Markov chain and estimate
the one-step transition matrix $P_{ij} = P(s_{t+1} = j \mid s_t = i)$.

In [ ]:
# Build transition counts from consecutive intervals in the same run
haz_sorted = hazard_intervals.sort_values(["run_id", "age_bin"]).copy()
haz_sorted["next_state"] = (
    haz_sorted.groupby("run_id")["p_state"].shift(-1)
)
trans = haz_sorted.dropna(subset=["next_state"])

# Count matrix
state_idx = {s: i for i, s in enumerate(STATE_LABELS)}
counts = np.zeros((3, 3))
for _, row in trans.iterrows():
    i = state_idx.get(row["p_state"],  -1)
    j = state_idx.get(row["next_state"], -1)
    if i >= 0 and j >= 0:
        counts[i, j] += 1

# Row-normalise
row_sums = counts.sum(axis=1, keepdims=True)
P = counts / np.where(row_sums == 0, 1, row_sums)

fig_C, ax_C = plt.subplots(figsize=(3.5, 3.0))
plot_state_transition_matrix(ax_C, P)
plt.tight_layout()
plt.show()

print("Transition matrix P:")
print(pd.DataFrame(P, index=STATE_LABELS, columns=STATE_LABELS).round(3))

## Panel D — Empirical vs. Simulated Run Durations

Simulate $N$ runs under the fitted Markov state process + age-dependent hazard.
Compare CCDF of simulated durations to empirical.

In [ ]:
rng = np.random.default_rng(42)

# Stationary distribution of the chain (for initial state sampling)
eigvals, eigvecs = np.linalg.eig(P.T)
stat = np.abs(eigvecs[:, np.argmax(np.abs(eigvals - 1.0) < 1e-8)])
stat = stat / stat.sum()

# Per-state hazard parameters from Panel B
def hazard_fn(a: float, state: str) -> float:
    p = state_params.get(state) or {"lambda_inf": 0.02, "mu": 1.0}
    return p["lambda_inf"] + p["mu"] / (A0 + a)

def simulate_runs(n_runs: int, dt: float = 1.0, max_t: float = 300.0) -> np.ndarray:
    durations = []
    for _ in range(n_runs):
        # Sample initial state
        state_i = rng.choice(len(STATE_LABELS), p=stat)
        age = 0.0
        while age < max_t:
            state = STATE_LABELS[state_i]
            h = hazard_fn(age, state)
            # Kill probability for this step
            if rng.random() < h * dt:
                break
            # Transition state
            state_i = rng.choice(len(STATE_LABELS), p=P[state_i])
            age += dt
        durations.append(age)
    return np.array(durations)

print(f"Simulating {N_SIM:,} runs …", end=" ")
sim_durations = simulate_runs(N_SIM)
print("done")
print(f"  Sim median: {np.median(sim_durations):.1f}s  |  Empirical median: {centroid_runs['duration_s'].median():.1f}s")

In [ ]:
fig_D, ax_D = plt.subplots(figsize=(4.5, 3.5))

plot_ccdf(ax_D, centroid_runs["duration_s"],
          label="Empirical", color="black", lw=1.5)
plot_ccdf(ax_D, sim_durations,
          label="Simulated (Markov+hazard)", color="#d6604d", ls="--", lw=1.2)

ax_D.set_xlabel("Run duration $T$ (s)")
ax_D.set_ylabel(r"$P(T \geq t)$")
ax_D.set_title(f"D — Empirical vs. simulated ($N$={N_SIM:,})")
ax_D.legend()
plt.tight_layout()
plt.show()

## Panel E — Hazard Ratios (Cox Model)

Fit a Cox proportional hazards model:
$$h(a, z_p, z_v) = h_0(a) \exp(\beta_p z_p + \beta_v z_v)$$

where $z_p$ is z-scored polarisation and $z_v$ is z-scored speed.

**HR < 1** for order means order is *protective* (suppresses termination).

> **Outstanding TODO**: Bootstrap CIs on HR estimates.
> AIC/BIC comparison (baseline vs. order model vs. order + speed model).

In [ ]:
try:
    from lifelines import CoxTimeVaryingFitter
    _lifelines_ok = True
except ImportError:
    print("lifelines not installed — skipping Cox model. Run: pip install lifelines")
    _lifelines_ok = False

if _lifelines_ok:
    df_cox = hazard_intervals.copy()

    # Z-score covariates
    df_cox["z_p"]     = (df_cox["p_group"]   - df_cox["p_group"].mean())   / df_cox["p_group"].std()
    df_cox["z_speed"] = (df_cox["speed_mps"] - df_cox["speed_mps"].mean()) / df_cox["speed_mps"].std()

    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        df_cox,
        id_col="run_id",
        event_col="event",
        start_col="age_bin",
        stop_col="age_bin_end",
        formula="z_p + z_speed",
    )
    ctv.print_summary()

    # Plot hazard ratios
    fig_E, ax_E = plt.subplots(figsize=(4.5, 2.5))
    hr = ctv.hazard_ratios_
    ci = ctv.confidence_intervals_
    names = ["Polarisation", "Speed"]
    y = [0, 1]

    ax_E.scatter(hr.values, y, color=["#4393c3", "#d6604d"], zorder=3, s=60)
    for i, idx in enumerate(hr.index):
        lo = ci.loc[idx, "95% lower-bound"]
        hi = ci.loc[idx, "95% upper-bound"]
        ax_E.plot([lo, hi], [y[i], y[i]], color=["#4393c3", "#d6604d"][i], lw=2)

    ax_E.axvline(1.0, color="k", lw=0.8, ls="--")
    ax_E.set_yticks(y); ax_E.set_yticklabels(names)
    ax_E.set_xlabel("Hazard ratio (HR)")
    ax_E.set_title("E — Cox model hazard ratios")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped Panel E — install lifelines")

## Statistical Inference — AIC/BIC Model Comparison

Compare:
1. Baseline: $h_0(a) = \lambda_\infty + \mu/(a_0+a)$
2. Order model: $h_0(a) \exp(\beta_p z_p)$
3. Full model: $h_0(a) \exp(\beta_p z_p + \beta_v z_v)$

> **TODO**: Implement log-likelihood for each model and compute AIC/BIC.

In [ ]:
def fit_inverse_age_hazard(age: np.ndarray, events: np.ndarray,
                           at_risk: np.ndarray, a0: float = A0,
                           beta_covariate: np.ndarray | None = None) -> dict:
    """
    MLE for the inverse-age hazard model with optional linear covariate.

    h(a, z) = [lambda_inf + mu/(a0+a)] * exp(beta * z)

    Returns dict with {lambda_inf, mu, beta (if covariate given), AIC, BIC, log_lik}.
    """
    n_params = 2 + (1 if beta_covariate is not None else 0)

    def neg_ll(theta: np.ndarray) -> float:
        lam, mu = theta[:2]
        if lam <= 0 or mu <= 0:
            return 1e9
        h0 = lam + mu / (a0 + age)
        if beta_covariate is not None:
            h = h0 * np.exp(theta[2] * beta_covariate)
        else:
            h = h0
        h = np.clip(h, 1e-12, None)
        # Poisson approximation: log-lik = sum(events * log(h) - at_risk * h)
        return -np.sum(events * np.log(h) - at_risk * h)

    x0 = [0.02, 1.0] + ([0.0] if beta_covariate is not None else [])
    res = minimize(neg_ll, x0=x0, method="Nelder-Mead")
    ll = -res.fun
    n = at_risk.sum()
    aic = 2 * n_params - 2 * ll
    bic = n_params * np.log(n) - 2 * ll

    out = {"lambda_inf": res.x[0], "mu": res.x[1], "log_lik": ll, "AIC": aic, "BIC": bic}
    if beta_covariate is not None:
        out["beta"] = res.x[2]
        out["HR"]   = float(np.exp(res.x[2]))
    return out


# Aggregate: events and at_risk per age_bin across all states
agg = (
    hazard_intervals
    .groupby("age_bin")["event"]
    .agg(events="sum", at_risk="count")
    .reset_index()
)
age_arr    = agg["age_bin"].values.astype(float)
events_arr = agg["events"].values.astype(float)
at_risk_arr= agg["at_risk"].values.astype(float)

# Model 0: baseline
m0 = fit_inverse_age_hazard(age_arr, events_arr, at_risk_arr)

# Model 1: + polarisation (run-level mean p — note: retrospective, descriptive only)
p_arr = hazard_intervals.groupby("age_bin")["p_group"].mean().values
p_arr = (p_arr - p_arr.mean()) / (p_arr.std() + 1e-9)
m1 = fit_inverse_age_hazard(age_arr, events_arr, at_risk_arr, beta_covariate=p_arr)

print("Model comparison:")
print(pd.DataFrame([m0, m1], index=["Baseline", "+ Polarisation"]).round(4))

## Assemble and Save Figure 4

In [ ]:
fig4 = plt.figure(figsize=(13, 8))
gs = fig4.add_gridspec(2, 3, hspace=0.45, wspace=0.38)

ax4_A = fig4.add_subplot(gs[0, 0])
ax4_B = fig4.add_subplot(gs[0, 1])
ax4_C = fig4.add_subplot(gs[0, 2])
ax4_D = fig4.add_subplot(gs[1, 0])
ax4_E = fig4.add_subplot(gs[1, 1:])

# A
plot_hazard(ax4_A, age_centres, haz_agg["h"].values,
            ci_lo=haz_agg["h_lo"].values, ci_hi=haz_agg["h_hi"].values,
            label="Empirical", color="black", fit_inverse_age=True)
ax4_A.set_title("A")

# B
for state in STATE_LABELS:
    sub = hazard_intervals.loc[hazard_intervals["p_state"] == state]
    hs = sub.groupby("age_bin")["event"].agg(events="sum", at_risk="count").assign(h=lambda d: d["events"]/d["at_risk"]).reset_index()
    plot_hazard(ax4_B, hs["age_bin"].values.astype(float), hs["h"].values,
                label=state.capitalize(), color=STATE_COLORS[state], fit_inverse_age=False)
ax4_B.legend(fontsize=8); ax4_B.set_title("B")

# C
plot_state_transition_matrix(ax4_C, P)

# D
plot_ccdf(ax4_D, centroid_runs["duration_s"], label="Empirical", color="black")
plot_ccdf(ax4_D, sim_durations, label="Simulated", color="#d6604d", ls="--")
ax4_D.legend(fontsize=8); ax4_D.set_title("D")
ax4_D.set_xlabel("Duration (s)"); ax4_D.set_ylabel(r"$P(\geq)$")

# E — Model comparison table
ax4_E.axis("off")
tbl_data = pd.DataFrame([m0, m1], index=["Baseline", "+Order"]).round(3)
ax4_E.table(cellText=tbl_data.values, colLabels=tbl_data.columns,
             rowLabels=tbl_data.index, loc="center", cellLoc="center")
ax4_E.set_title("E — Model comparison (AIC/BIC)", pad=30)

fig4.suptitle("Figure 4 — Hazard Mechanism", fontsize=13)
save_figure(fig4, "figure4_hazard_mechanism")
plt.show()